In [ ]:
### Comparison 
## Gordevio 

In [7]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

# Function to read ASCII .max file
def read_ascii(filename):
    with open(filename, 'r') as f:
        header = {}
        for _ in range(6):  # First 6 lines are header
            line = f.readline()
            key, value = line.strip().split()
            header[key] = float(value)
        data = np.loadtxt(f)
    return header, data

# Threshold to define 'flooded'
flood_threshold = 0.10  # 1 cm depth = flooded
rainfall_steps = range(10, 111, 10)  # 10 to 110 mm/h

# Root directory where all DG2 and FV1 folders are located
root_dir = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Gordevio_2m"

# Output folder (different from root_dir, as requested)
output_folder = "/storage/homefs/ge24z347/LISFLOOD_FP_outputs/Gordevio_comparison_solvers"
os.makedirs(output_folder, exist_ok=True)

# Define color map and legend
colors = ['white', 'green', 'blue', 'red']
cmap = ListedColormap(colors)

legend_labels = [
    mpatches.Patch(color='white', label='Dry in both'),
    mpatches.Patch(color='green', label='Flood in both'),
    mpatches.Patch(color='blue', label='Only DG2 floods'),
    mpatches.Patch(color='red', label='Only FV1 floods')
]

# Loop through rainfall scenarios
for intensity in rainfall_steps:
    # Folder and file names
    fv1_folder = os.path.join(root_dir, f"Gordevio_2m_{intensity}_fv1-gpu")
    dg2_folder = os.path.join(root_dir, f"Gordevio_2m_{intensity}_dg2-gpu")
    filename = f"Gordevio_2m_{intensity}_1h_{intensity}.max"

    fv1_file = os.path.join(fv1_folder, filename)
    dg2_file = os.path.join(dg2_folder, filename)

    if not os.path.exists(dg2_file) or not os.path.exists(fv1_file):
        print(f"⚠️ Missing file for {intensity} mm/h. Skipping.")
        continue

    # Read both .max files
    header_dg2, dg2 = read_ascii(dg2_file)
    header_fv1, fv1 = read_ascii(fv1_file)

    # Create masks and comparison
    dg2_flood = dg2 > flood_threshold
    fv1_flood = fv1 > flood_threshold

    comparison = np.zeros_like(dg2, dtype=np.uint8)
    comparison[(dg2_flood) & (fv1_flood)] = 1
    comparison[(dg2_flood) & (~fv1_flood)] = 2
    comparison[(~dg2_flood) & (fv1_flood)] = 3

    # Plot
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(comparison, cmap=cmap)
    ax.set_title(f"Gordevio - {intensity} mm/h\nFlood Extent Comparison: DG2 vs FV1")
    ax.axis('off')
    ax.legend(handles=legend_labels, loc='lower center', bbox_to_anchor=(0.5, -0.1),
              ncol=2, fancybox=True, shadow=True)

    # Save figure to your custom folder
    output_path = os.path.join(output_folder, f"Gordevio_DG2_vs_FV1_{intensity}mm_per_h_comparison.png")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()
    print(f"✅ Saved: {output_path}")

✅ Saved: /storage/homefs/ge24z347/LISFLOOD_FP_outputs/Gordevio_comparison_solvers/Gordevio_DG2_vs_FV1_10mm_per_h_comparison.png
✅ Saved: /storage/homefs/ge24z347/LISFLOOD_FP_outputs/Gordevio_comparison_solvers/Gordevio_DG2_vs_FV1_20mm_per_h_comparison.png
✅ Saved: /storage/homefs/ge24z347/LISFLOOD_FP_outputs/Gordevio_comparison_solvers/Gordevio_DG2_vs_FV1_30mm_per_h_comparison.png
✅ Saved: /storage/homefs/ge24z347/LISFLOOD_FP_outputs/Gordevio_comparison_solvers/Gordevio_DG2_vs_FV1_40mm_per_h_comparison.png
✅ Saved: /storage/homefs/ge24z347/LISFLOOD_FP_outputs/Gordevio_comparison_solvers/Gordevio_DG2_vs_FV1_50mm_per_h_comparison.png
✅ Saved: /storage/homefs/ge24z347/LISFLOOD_FP_outputs/Gordevio_comparison_solvers/Gordevio_DG2_vs_FV1_60mm_per_h_comparison.png
✅ Saved: /storage/homefs/ge24z347/LISFLOOD_FP_outputs/Gordevio_comparison_solvers/Gordevio_DG2_vs_FV1_70mm_per_h_comparison.png
✅ Saved: /storage/homefs/ge24z347/LISFLOOD_FP_outputs/Gordevio_comparison_solvers/Gordevio_DG2_vs_FV1_80

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
import requests
from PIL import Image
from io import BytesIO

# Function to read ASCII .max file
def read_ascii(filename):
    with open(filename, 'r') as f:
        header = {}
        for _ in range(6):  # First 6 lines are header
            line = f.readline()
            key, value = line.strip().split()
            header[key] = float(value)
        data = np.loadtxt(f)
    return header, data

# Function to fetch Swisstopo WMS background
def fetch_swisstopo_wms_background(bounds):
    wms_url = "https://wms.geo.admin.ch/"
    bbox = f"{bounds['left']},{bounds['bottom']},{bounds['right']},{bounds['top']}"
    wms_params = {
        "SERVICE": "WMS",
        "REQUEST": "GetMap",
        "VERSION": "1.3.0",
        "LAYERS": "ch.swisstopo.swisstlm3d-karte-grau",
        "FORMAT": "image/png",
        "TRANSPARENT": "false",
        "WIDTH": 1024,
        "HEIGHT": 768,
        "CRS": "EPSG:2056",
        "BBOX": bbox
    }

    response = requests.get(wms_url, params=wms_params)
    if response.status_code == 200:
        return Image.open(BytesIO(response.content))
    else:
        print("⚠️ Failed to fetch Swisstopo WMS basemap")
        return None

# Threshold to define 'flooded'
flood_threshold = 0.10
rainfall_steps = range(10, 111, 10)

# Root and output folders
root_dir = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Gordevio_2m"
output_folder = "/storage/homefs/ge24z347/LISFLOOD_FP_outputs/Gordevio_comparison_solvers"
os.makedirs(output_folder, exist_ok=True)

# Color map for only flood classes (exclude dry=0)
colors = ['green', 'blue', 'red']  # only values 1, 2, 3 will be plotted
cmap = ListedColormap(colors)

# Legend (still includes dry class for reference)
legend_labels = [
    mpatches.Patch(color='green', label='Flood in both'),
    mpatches.Patch(color='blue', label='Only DG2 floods'),
    mpatches.Patch(color='red', label='Only FV1 floods'),
    mpatches.Patch(color='lightgray', label='Dry in both (background)')
]

# Get DEM bounds from a sample .max file
sample_header, _ = read_ascii(os.path.join(
    root_dir, "Gordevio_2m_10_dg2-gpu", "Gordevio_2m_10_1h_10.max"))
ncols = int(sample_header["ncols"])
nrows = int(sample_header["nrows"])
xll = sample_header["xllcorner"]
yll = sample_header["yllcorner"]
cellsize = sample_header["cellsize"]

bounds = {
    "left": xll,
    "right": xll + ncols * cellsize,
    "bottom": yll,
    "top": yll + nrows * cellsize
}

# Fetch the basemap once
basemap_img = fetch_swisstopo_wms_background(bounds)

if basemap_img is None:
    print(" Aborting. Could not retrieve Swisstopo background.")
    exit()

# Main loop for all intensities
for intensity in rainfall_steps:
    fv1_folder = os.path.join(root_dir, f"Gordevio_2m_{intensity}_fv1-gpu")
    dg2_folder = os.path.join(root_dir, f"Gordevio_2m_{intensity}_dg2-gpu")
    filename = f"Gordevio_2m_{intensity}_1h_{intensity}.max"
    fv1_file = os.path.join(fv1_folder, filename)
    dg2_file = os.path.join(dg2_folder, filename)

    if not os.path.exists(dg2_file) or not os.path.exists(fv1_file):
        print(f"⚠️ Missing file for {intensity} mm/h. Skipping.")
        continue

    # Load and process data
    header_dg2, dg2 = read_ascii(dg2_file)
    header_fv1, fv1 = read_ascii(fv1_file)
    dg2_flood = dg2 > flood_threshold
    fv1_flood = fv1 > flood_threshold

    comparison = np.zeros_like(dg2, dtype=np.uint8)
    comparison[(dg2_flood) & (fv1_flood)] = 1
    comparison[(dg2_flood) & (~fv1_flood)] = 2
    comparison[(~dg2_flood) & (fv1_flood)] = 3

    # Mask dry areas (0) to make them transparent
    comparison_masked = np.where(comparison == 0, np.nan, comparison)

    # Plot
    fig, ax = plt.subplots(figsize=(12, 10))

    # Show Swisstopo background
    ax.imshow(basemap_img, extent=(bounds['left'], bounds['right'], bounds['bottom'], bounds['top']),
              interpolation="none", zorder=0)

    # Overlay flood difference map with transparency on dry areas
    ax.imshow(comparison_masked, cmap=cmap,
              extent=(bounds['left'], bounds['right'], bounds['bottom'], bounds['top']),
              interpolation="none", zorder=1)

    ax.set_title(f"Gordevio - {intensity} mm/h\nFlood Extent Comparison: DG2 vs FV1", fontsize=14)
    ax.axis('off')
    ax.legend(handles=legend_labels, loc='lower center', bbox_to_anchor=(0.5, -0.1),
              ncol=2, fancybox=True, shadow=True)

    output_path = os.path.join(output_folder, f"Gordevio_DG2_vs_FV1_{intensity}mm_per_h_comparison.png")
    plt.tight_layout()
    plt.savefig(output_path, dpi=800)
    plt.close()
    print(f" Saved: {output_path}")

✅ Saved: /storage/homefs/ge24z347/LISFLOOD_FP_outputs/Gordevio_comparison_solvers/Gordevio_DG2_vs_FV1_10mm_per_h_comparison.png
✅ Saved: /storage/homefs/ge24z347/LISFLOOD_FP_outputs/Gordevio_comparison_solvers/Gordevio_DG2_vs_FV1_20mm_per_h_comparison.png
✅ Saved: /storage/homefs/ge24z347/LISFLOOD_FP_outputs/Gordevio_comparison_solvers/Gordevio_DG2_vs_FV1_30mm_per_h_comparison.png
✅ Saved: /storage/homefs/ge24z347/LISFLOOD_FP_outputs/Gordevio_comparison_solvers/Gordevio_DG2_vs_FV1_40mm_per_h_comparison.png
✅ Saved: /storage/homefs/ge24z347/LISFLOOD_FP_outputs/Gordevio_comparison_solvers/Gordevio_DG2_vs_FV1_50mm_per_h_comparison.png
✅ Saved: /storage/homefs/ge24z347/LISFLOOD_FP_outputs/Gordevio_comparison_solvers/Gordevio_DG2_vs_FV1_60mm_per_h_comparison.png
✅ Saved: /storage/homefs/ge24z347/LISFLOOD_FP_outputs/Gordevio_comparison_solvers/Gordevio_DG2_vs_FV1_70mm_per_h_comparison.png
✅ Saved: /storage/homefs/ge24z347/LISFLOOD_FP_outputs/Gordevio_comparison_solvers/Gordevio_DG2_vs_FV1_80